# Week 09 - Wikipedia

[Brian C. Keegan, Ph.D.](http://brianckeegan.com/)  
[Assistant Professor, Department of Information Science](https://www.colorado.edu/cmci/people/information-science/brian-c-keegan)  
University of Colorado Boulder  

Copyright and distributed under an [MIT License](https://opensource.org/licenses/MIT)  

In [86]:
import numpy as np
np.set_printoptions(suppress=True)

import pandas as pd
idx = pd.IndexSlice
pd.options.display.max_columns = 100

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sb

import requests
import wikifunctions as wf

We will use the English Wikipedia article for Queen [Elizabeth II](https://en.wikipedia.org/wiki/Elizabeth_II) and [this revision](https://en.wikipedia.org/w/index.php?title=Elizabeth_II&oldid=1016271620) from April 6, 2021 as examples.

### Get all page revisions

Get a DataFrame of every revision made to the article since mid-2001.

In [ ]:
revs_df = wf.get_all_page_revisions('Elizabeth II')

revs_df.head()

In [ ]:
len(revs_df)

In [ ]:
len(revs_df['sha1'].unique())

In [ ]:
revs_df[revs_df['sha1'] == 'da39a3ee5e6b4b0d3255bfef95601890afd80709']

#### Analysis example
Count the number of unique revisions per month.

In [ ]:
revs_df.groupby(pd.Grouper(key='timestamp',freq='MS')).agg({'sha1':'nunique'})['sha1'].plot()

#### Another example

In [ ]:
revs_df['user'].value_counts().head(10)

### Get pageviews

Get the number of times the article was accessed across web and mobile since July 2015.

In [ ]:
pvs1 = wf.get_pageviews('Charli XCX')
pvs2 = wf.get_pageviews('Troye Sivan')
pvs3 = wf.get_pageviews('Taylor Swift')
pvs1.head()

#### Analysis example

In [ ]:
f,ax = plt.subplots(1,1,figsize=(12,6))
pvs1.plot(ax=ax,label='Charli',color='tab:red')
pvs2.plot(ax=ax,label='Troye',color='tab:blue')
pvs3.plot(ax=ax,label='Swift',color='tab:green')
# ax.axvline(pd.Timestamp('2022-09-08'),color='r')
# ax.set_yscale('log')
ax.legend()

In [ ]:
pd.Series(pvs2.index).apply(lambda x:x.isoweekday())

### Get current page content

In [ ]:
page_content = wf.get_page_content('Charli XCX')

print(page_content[:1000])

#### Analysis example

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

current_words = word_tokenize(page_content)
current_words = [word.lower() for word in current_words if word.isalpha() and word not in stopwords.words('english')]
print(len(current_words))

### Get revision content

In [ ]:
revision_content = wf.get_revision_content(revid='212642648')
revision_content[:1000]

#### Analysis example

In [ ]:
revision_words = word_tokenize(revision_content)
revision_words = [word.lower() for word in revision_words if word.isalpha() and word not in stopwords.words('english')]
print(len(revision_words))

#### Another example

In [ ]:
from scipy import spatial


In [ ]:
from scipy import spatial

current_words_s = pd.Series(current_words).value_counts()
revision_words_s = pd.Series(revision_words).value_counts()

compare_df = pd.concat(
    {'Current':current_words_s,
     'Revision':revision_words_s
    },
     axis=1
     )
compare_df.fillna(0,inplace=True)

1 - spatial.distance.cosine(compare_df['Current'], compare_df['Revision'])

In [ ]:
compare_df

### Get current page outlinks

In [ ]:
current_outlinks = wf.get_page_outlinks('Double Asteroid Redirection Test')
current_outlinks[:10]

In [ ]:
current_outlinks = wf.get_page_outlinks('Zweiter Weltkrieg',endpoint='de.wikipedia.org/w/api.php')
current_outlinks[:10]

### Get revision outlinks

In [ ]:
revision_outlinks = wf.get_revision_outlinks(revid='821522967')
revision_outlinks[:5]

#### Analysis example

In [ ]:
current_outlinks_s = pd.Series(current_outlinks).value_counts()
revision_outlinks_s = pd.Series(revision_outlinks).value_counts()

compare_outlinks_df = pd.concat(
    {'Current':current_outlinks_s,
     'Revision':revision_outlinks_s
    },
     axis=1
     )
compare_outlinks_df.fillna(0,inplace=True)

1 - spatial.distance.cosine(compare_outlinks_df['Current'], compare_outlinks_df['Revision'])

### Get current page externallinks

In [ ]:
current_externallinks = wf.get_page_externallinks('Elizabeth II')
current_externallinks[:5]

### Get revision externallinks

In [ ]:
revision_externallinks = wf.get_revision_externallinks(revid='1016271620')
revision_externallinks[:5]

#### Analysis example

In [ ]:
revision_domains

In [ ]:
from urllib.parse import urlparse

current_domains = [urlparse(link).netloc for link in current_externallinks]
revision_domains = [urlparse(link).netloc for link in revision_externallinks]

current_domains_s = pd.Series(current_domains).value_counts()
revision_domains_s = pd.Series(revision_domains).value_counts()

compare_domains_df = pd.concat(
    {'Current':current_domains_s,
     'Revision':revision_domains_s
    },
     axis=1
     )
compare_domains_df.fillna(0,inplace=True)

1 - spatial.distance.cosine(compare_domains_df['Current'], compare_domains_df['Revision'])

In [ ]:
compare_domains_df

### Get interlanguage links

In [ ]:
ill_d = wf.get_interlanguage_links('Arab Spring')
ill_d

#### Analysis example

In [ ]:
revs_en_df = wf.get_all_page_revisions(
    page_title = ill_d['en'],
    endpoint = 'en.wikipedia.org/w/api.php'
)

In [ ]:
revs_ar_df = wf.get_all_page_revisions(
    page_title = ill_d['ar'],
    endpoint = 'ar.wikipedia.org/w/api.php'
)

In [ ]:
monthly_revs_en = revs_en_df.groupby(pd.Grouper(key='timestamp',freq='m')).agg({'sha1':'nunique'})['sha1']
monthly_revs_ar = revs_ar_df.groupby(pd.Grouper(key='timestamp',freq='m')).agg({'sha1':'nunique'})['sha1']

monthly_revs = pd.concat(
    {'English':monthly_revs_en,
     'Arabic':monthly_revs_ar
    },
    axis = 1
)

rev_ratio = monthly_revs#['English'] / monthly_revs['Chinese']

f,ax = plt.subplots(figsize=(12,6))
rev_ratio.plot(ax=ax)
ax.set_yscale('log')
# ax.set_ylabel('English to Arabic revision ratio')
ax.axhline(1,ls='--',color='k')

### Get category memberships

In [ ]:
e2_memberships = wf.get_category_memberships('Elizabeth II')
e2_memberships[:10]

### Get category members

In [ ]:
woman_rulers = wf.get_category_members('20th-century women rulers',depth=0)
woman_rulers[:10]

### Get user information

In [ ]:
wf.get_user_info(['Madcoverboy'])

### Get user contributions

In [ ]:
drkay_contribs = wf.get_user_contributions(
    username = 'Madcoverboy',
    start = '2022-09-01',
    stop = '2023-10-17'
)

print(len(drkay_contribs))

drkay_contribs.head()

#### Analysis example

In [ ]:
drkay_contribs['ns'].value_counts()

#### Another example

In [ ]:
ax = drkay_contribs.groupby(pd.Grouper(key='timestamp',freq='h')).agg({'revid':'nunique'})['revid'].plot()
ax.axvline(pd.Timestamp('2022-09-08'),color='red')

## World War II case

In [ ]:
en_outlinks = wf.get_page_outlinks('World War II')
en_outlinks[:10]

In [ ]:
ww2_en = 'World War II'

ww2_en_ill = wf.get_interlanguage_links(ww2_en)

ww2_de = ww2_en_ill['de']
ww2_ja = ww2_en_ill['ja']

In [ ]:
ww2_en_outlinks = wf.get_page_outlinks(ww2_en,endpoint='en.wikipedia.org/w/api.php')

In [ ]:
ww2_de_outlinks = wf.get_page_outlinks(ww2_de,endpoint='de.wikipedia.org/w/api.php')
ww2_ja_outlinks = wf.get_page_outlinks(ww2_ja,endpoint='ja.wikipedia.org/w/api.php')

In [ ]:
ww2_de_outlinks_ill = {}

for outlink in ww2_de_outlinks:
    try:
        ww2_de_outlinks_ill[outlink] = wf.get_interlanguage_links(outlink,endpoint='de.wikipedia.org/w/api.php')
    except KeyboardInterrupt:
        break
    except:
        print(outlink)
        pass

In [ ]:
ww2_ja_outlinks_ill = {}

for outlink in ww2_ja_outlinks:
    try:
        ww2_ja_outlinks_ill[outlink] = wf.get_interlanguage_links(outlink,endpoint='ja.wikipedia.org/w/api.php')
    except KeyboardInterrupt:
        break
    except:
        print(outlink)
        pass

In [ ]:
ww2_de_outlinks_ill_en = [link_dict.get('en',outlink) for outlink,link_dict in ww2_de_outlinks_ill.items()]
ww2_ja_outlinks_ill_en = [link_dict.get('en',outlink) for outlink,link_dict in ww2_ja_outlinks_ill.items()]

In [ ]:
set(ww2_en_outlinks) & set(ww2_de_outlinks_ill_en)

In [ ]:
set(ww2_en_outlinks) - set(ww2_de_outlinks_ill_en)

In [ ]:
set(ww2_de_outlinks_ill_en) - set(ww2_en_outlinks)

In [ ]:
set(ww2_ja_outlinks_ill_en) & set(ww2_en_outlinks)

In [ ]:
set(ww2_en_outlinks) - set(ww2_ja_outlinks_ill_en)

In [ ]:
set(ww2_ja_outlinks_ill_en) - set(ww2_en_outlinks)

In [ ]:
set(ww2_de_outlinks_ill_en) & set(ww2_ja_outlinks_ill_en)

In [ ]:
set(ww2_de_outlinks_ill_en) - set(ww2_ja_outlinks_ill_en)

In [ ]:
set(ww2_ja_outlinks_ill_en) - set(ww2_de_outlinks_ill_en)

## Practice writing API requests

The functions above packaged writing API requests and parsing their output for you. Let's practice writing an API request against the Wikipedia API yourself.

### Get links on page

Wikipedia has a "[query](https://www.mediawiki.org/wiki/API:Query)" API endpoint that contains a number of functions. Let's focus on the [getting the links](https://www.mediawiki.org/wiki/API:Links) on the current page. This is probably easier and more reliable than writing your own parser from scratch (Inspect, find tags, write BeautifulSoup code, etc.). The documentation includes a code example on how to use `requests`.

This will request 20 links from the "University of Colorado Boulder" page in JSON format.

In [7]:
wiki_session = requests.Session()

enwiki_api_url = "https://en.wikipedia.org/w/api.php"

wiki_params = {
    "action": "query",
    "prop": "links",
    "titles": "Freddie Freeman",
    "pllimit": "20",
    "format": "json",
}

wiki_request = wiki_session.get(url=enwiki_api_url, params=wiki_params)
wiki_json = wiki_request.json()

wiki_json

{'continue': {'plcontinue': '23676613|0|2023_World_Baseball_Classic',
  'continue': '||'},
 'query': {'pages': {'23676613': {'pageid': 23676613,
    'ns': 0,
    'title': 'Freddie Freeman',
    'links': [{'ns': 0, 'title': '1929 Brooklyn Robins season'},
     {'ns': 0, 'title': '2004 Los Angeles Dodgers season'},
     {'ns': 0, 'title': '2007 Major League Baseball draft'},
     {'ns': 0, 'title': '2010 Major League Baseball season'},
     {'ns': 0, 'title': '2013 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2014 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2015 Atlanta Braves season'},
     {'ns': 0, 'title': '2017 World Baseball Classic'},
     {'ns': 0, 'title': '2018 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2018 Major League Baseball Home Run Derby'},
     {'ns': 0, 'title': '2019 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2021 Atlanta Braves season'},
     {'ns': 0, 'title': '2021 Major League Basebal

You'll need to navigate into a somewhat deep JSON document to get your results out.

Frustratingly, there is a numeric string for the page_id unique to each page that you'll need to retrieve to navigate deeper into the dictionary.

In [8]:
list(wiki_json['query']['pages'].keys())[0]

'23676613'

Work through the `wiki_json` response and add the list of links hiding inside to `title_list`.

In [14]:
def parse_json_links_to_list(wiki_json):
    title_list = []
    
    # If there are results from the query
    if 'query' in wiki_json:
    
        # Get the pageid
        page_id = list(wiki_json['query']['pages'].keys())[0]
    
        # Get the links
        links = wiki_json['query']['pages'][page_id]['links']
    
        # For each link dictionary element
        for l in links:
    
            # Get the title and append it to the list
            title_list.append(l['title'])
    
    return title_list

In [15]:
parse_json_links_to_list(wiki_json)

['1929 Brooklyn Robins season',
 '2004 Los Angeles Dodgers season',
 '2007 Major League Baseball draft',
 '2010 Major League Baseball season',
 '2013 Major League Baseball All-Star Game',
 '2014 Major League Baseball All-Star Game',
 '2015 Atlanta Braves season',
 '2017 World Baseball Classic',
 '2018 Major League Baseball All-Star Game',
 '2018 Major League Baseball Home Run Derby',
 '2019 Major League Baseball All-Star Game',
 '2021 Atlanta Braves season',
 '2021 Major League Baseball All-Star Game',
 '2021 Major League Baseball season',
 '2021 National League Championship Series',
 '2021 National League Division Series',
 '2021 World Series',
 '2022 Colorado Rockies season',
 '2022 Major League Baseball season',
 '2023 Major League Baseball All-Star Game',
 '2023 World Baseball Classic',
 '2024 Los Angeles Dodgers season',
 '2024 Major League Baseball All-Star Game',
 '2024 NLCS',
 '2024 NLDS',
 '2024 National League Championship Series',
 '2024 National League Division Series',
 '2

### Exercise 1: Change the API request

Change API request to get the maximum number of items per request. How many is the maximum? (Hint: read the documentation!)

In [12]:
wiki_session = requests.Session()

enwiki_api_url = "https://en.wikipedia.org/w/api.php"

wiki_params = {
    "action": "query",
    "prop": "links",
    "titles": "Freddie Freeman",
    "pllimit": "501",
    "format": "json",
}

wiki_request = wiki_session.get(url=enwiki_api_url, params=wiki_params)
wiki_json = wiki_request.json()

wiki_json

{'continue': {'plcontinue': '23676613|0|Ray_Knight', 'continue': '||'},
 'warnings': {'links': {'*': 'The value "501" for parameter "pllimit" must be between 1 and 500.'}},
 'query': {'pages': {'23676613': {'pageid': 23676613,
    'ns': 0,
    'title': 'Freddie Freeman',
    'links': [{'ns': 0, 'title': '1929 Brooklyn Robins season'},
     {'ns': 0, 'title': '2004 Los Angeles Dodgers season'},
     {'ns': 0, 'title': '2007 Major League Baseball draft'},
     {'ns': 0, 'title': '2010 Major League Baseball season'},
     {'ns': 0, 'title': '2013 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2014 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2015 Atlanta Braves season'},
     {'ns': 0, 'title': '2017 World Baseball Classic'},
     {'ns': 0, 'title': '2018 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2018 Major League Baseball Home Run Derby'},
     {'ns': 0, 'title': '2019 Major League Baseball All-Star Game'},
     {'ns': 0, 'title':

### Exercise 2: Retrieve additional data

The query:links request we made above may include a "continue" key if there are more results than the limit allows. Update the `wiki_params` to request the next set of results using the "continue" key. (Hint: [read the documentation](https://www.mediawiki.org/wiki/API:Continue)!)

In [95]:
wiki_session = requests.Session()

enwiki_api_url = "https://en.wikipedia.org/w/api.php"

wiki_params = {
    "action": "query",
    "prop": "links",
    "titles": "Freddie Freeman",
    "plnamespace":"0",
    "pllimit": "500",
    "format": "json",
    "plcontinue":{}
}

wiki_request = wiki_session.get(url=enwiki_api_url, params=wiki_params)
wiki_json = wiki_request.json()

wiki_json

{'continue': {'plcontinue': '23676613|0|Rawlings_Gold_Glove_Award',
  'continue': '||'},
 'query': {'pages': {'23676613': {'pageid': 23676613,
    'ns': 0,
    'title': 'Freddie Freeman',
    'links': [{'ns': 0, 'title': '1929 Brooklyn Robins season'},
     {'ns': 0, 'title': '2004 Los Angeles Dodgers season'},
     {'ns': 0, 'title': '2007 Major League Baseball draft'},
     {'ns': 0, 'title': '2010 Major League Baseball season'},
     {'ns': 0, 'title': '2013 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2014 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2015 Atlanta Braves season'},
     {'ns': 0, 'title': '2017 World Baseball Classic'},
     {'ns': 0, 'title': '2018 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2018 Major League Baseball Home Run Derby'},
     {'ns': 0, 'title': '2019 Major League Baseball All-Star Game'},
     {'ns': 0, 'title': '2021 Atlanta Braves season'},
     {'ns': 0, 'title': '2021 Major League Baseball 

### Exercise 3: Write a function

Abstract the code above into a function that accepts an article name string input and returns all the outlinks (>500) as a list of strings. Test the function on a few different pages, particularly pages that have non-standard characters (Beyoncé, Teoscar Hernández, *etc*.)

In [124]:
def get_all_page_links(page_title):
    links = []

    # Set up query
    wiki_session = requests.Session()

    enwiki_api_url = "https://en.wikipedia.org/w/api.php"

    pl_continue = {}
    
    wiki_params = {
        "action": "query",
        "prop": "links",
        "titles": page_title,
        "plnamespace":"0",
        "pllimit": "500",
        "format": "json",
        'plcontinue':pl_continue
    }

    i = 0
    # While there are more links to get, get more links
    while True:

        # Get links
        result = wiki_session.get(url=enwiki_api_url, params=wiki_params).json()

        if 'query' in result:
            # Get the pageid
            page_id = list(result['query']['pages'].keys())[0]
        
            # Get the links
            result_links = result['query']['pages'][page_id]['links']
        
            # For each link dictionary element
            for l in result_links:
        
                # Get the title and append it to the list
                links.append(l['title'])

            if 'continue' in result:
                pl_continue = result['continue']['plcontinue']
            
            # When we're out of links, don't get more
            if 'batchcomplete' in result:
                break

        wiki_params['plcontinue'] = pl_continue
            
    return links

In [125]:
len(get_all_page_links('Freddie Freeman'))

631

### Exercise 4: Write a function for other wiki projects

The functions above used a `enwiki_api_url` that pointed at a URL "https://en.wikipedia.org/w/api.php" where "en" refers to the English-language version of Wikipedia. Retrieve the outlinks from a non-English (es.wikipedia.org for Spanish, *etc*.) or non-Wikipedia page (en.wiktionary.org for Wiktionary, *etc*.). Test with a page title that exists on that project.

In [61]:
wiki_session = requests.Session()

eswiki_api_url = "https://en.wiktionary.org/w/api.php"

wiki_params = {
    "action": "query",
    "prop": "links",
    "titles": "witching hour",
    "plnamespace":"0",
    "pllimit": "500",
    "format": "json",
    "plcontinue":{}
}

wiki_request = wiki_session.get(url=eswiki_api_url, params=wiki_params)
wiki_json = wiki_request.json()

wiki_json

{'batchcomplete': '',
 'query': {'pages': {'104549': {'pageid': 104549,
    'ns': 0,
    'title': 'witching hour',
    'links': [{'ns': 0, 'title': 'Geisterstunde'},
     {'ns': 0, 'title': 'a.m.'},
     {'ns': 0, 'title': 'active'},
     {'ns': 0, 'title': 'ascribe'},
     {'ns': 0, 'title': 'associate'},
     {'ns': 0, 'title': 'baby'},
     {'ns': 0, 'title': 'bad luck'},
     {'ns': 0, 'title': 'being'},
     {'ns': 0, 'title': 'child'},
     {'ns': 0, 'title': 'cry'},
     {'ns': 0, 'title': 'dead of night'},
     {'ns': 0, 'title': 'demon'},
     {'ns': 0, 'title': 'early'},
     {'ns': 0, 'title': 'evening'},
     {'ns': 0, 'title': 'evil hour'},
     {'ns': 0, 'title': 'expire'},
     {'ns': 0, 'title': 'final'},
     {'ns': 0, 'title': 'fretful'},
     {'ns': 0, 'title': 'fuss'},
     {'ns': 0, 'title': 'gluvo doba'},
     {'ns': 0, 'title': 'godzina duchów'},
     {'ns': 0, 'title': 'great'},
     {'ns': 0, 'title': 'high'},
     {'ns': 0, 'title': 'hodina duchů'},
     {'ns'

### Exercise 5: Get other kinds of pageview data

We have been talking to the MediaWiki API endpoint to get the data. The Wikimedia Foundation has created other APIs to support researchers. The pageviews API endpoint is especially powerful but uses a different infrastructure and therefore has a different endpoint format: you need to [read the documentation](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html). 

This API endpoint is a little more cranky about just giving data away: you [have to include a user-agent identification](https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/documentation/access-policy.html#client-identification).

> "How do I include user-agent strings in a request?"

Look at Weeks 2-4 or [find some documentation](https://stackoverflow.com/a/10606260/1574687)!

Experiment with different functionalities with this API. Write a function to retrieve some data from this API endpoint.

In [71]:
user_agent = {'user-agent':'brian.keegan@colorado.edu'}
pv_json = requests.get("https://wikimedia.org/api/rest_v1/metrics/pageviews/top-per-country/US/all-access/2024/01/01",headers=user_agent).json()

pd.DataFrame(pv_json['items'][0]['articles'])

,article,project,views_ceil,rank
0,Main_Page,en.wikipedia,1866400,1
1,Special:Search,en.wikipedia,446500,2
2,Lynyrd_Skynyrd,en.wikipedia,397500,3
3,Elle_King,en.wikipedia,327500,4
4,Paul_Anka,en.wikipedia,286000,5
...,...,...,...,...
995,Brandon_Burlsworth,en.wikipedia,6500,996
996,Jordan_Belfort,en.wikipedia,6500,997
997,Brooklyn,en.wikipedia,6500,998
998,The_Man_Who_Stepped_into_Yesterday,en.wikipedia,6500,999


### Exercise 6: Get data from Fandom

Fandom is a collection of wikis about specific fandoms: [Star Trek](https://memory-alpha.fandom.com/), [Star Wars](https://starwars.fandom.com/), [Harry Potter](https://harrypotter.fandom.com/), [Avatar: The Last Airbender](https://avatar.fandom.com/wiki/), *etc*.

Despite the differences in interfaces (and so, so many ads), these pages are all built on top of the same MediaWiki software that powers Wikipedia. So we can access their MediaWiki APIs to get data. The endpoints are slightly different: `https://memory-alpha.fandom.com/api.php`

Update and use your function to get the links present on a Fandom page.

In [72]:
wiki_session = requests.Session()

startrek_wiki_api_url = "https://memory-alpha.fandom.com/api.php"

wiki_params = {
    "action": "query",
    "prop": "links",
    "titles": "Jean-Luc Picard",
    "plnamespace":"0",
    "pllimit": "500",
    "format": "json",
    "plcontinue":{}
}

wiki_request = wiki_session.get(url=startrek_wiki_api_url, params=wiki_params)
wiki_json = wiki_request.json()

wiki_json

{'continue': {'plcontinue': "289509|0|Journey's_End_(episode)",
  'continue': '||'},
 'query': {'pages': {'289509': {'pageid': 289509,
    'ns': 0,
    'title': 'Jean-Luc Picard',
    'links': [{'ns': 0, 'title': '10 Forward Avenue'},
     {'ns': 0, 'title': '11001001 (episode)'},
     {'ns': 0, 'title': '13 April'},
     {'ns': 0, 'title': '14 April'},
     {'ns': 0, 'title': '1893'},
     {'ns': 0, 'title': '1986 (production)'},
     {'ns': 0, 'title': '1987 (production)'},
     {'ns': 0, 'title': '1997 (production)'},
     {'ns': 0, 'title': '19th century'},
     {'ns': 0, 'title': '1 May'},
     {'ns': 0, 'title': '2024'},
     {'ns': 0, 'title': '2063'},
     {'ns': 0, 'title': '20th century'},
     {'ns': 0, 'title': '2305'},
     {'ns': 0, 'title': '2310s'},
     {'ns': 0, 'title': '2322'},
     {'ns': 0, 'title': '2323'},
     {'ns': 0, 'title': '2327'},
     {'ns': 0, 'title': '2328'},
     {'ns': 0, 'title': '2333'},
     {'ns': 0, 'title': '2339'},
     {'ns': 0, 'title': '2

### Exercise 7: Get data from Ballotpedia

Super relevant for our final project, the [Ballotpedia](https://ballotpedia.org/Main_Page) is an encyclopedia of American politics and is also built on top of MediaWiki. Its API endpoint is `https://ballotpedia.org/wiki/api.php` 

Get the outlinks present on a page for a candidate or topic on Ballotpedia: [Judy Amabile](https://ballotpedia.org/Judy_Amabile)

In [73]:
wiki_session = requests.Session()

ballotpedia_wiki_api_url = "https://ballotpedia.org/wiki/api.php"

wiki_params = {
    "action": "query",
    "prop": "links",
    "titles": "Judy Amabile",
    "plnamespace":"0",
    "pllimit": "500",
    "format": "json",
    "plcontinue":{}
}

wiki_request = wiki_session.get(url=ballotpedia_wiki_api_url, params=wiki_params)
wiki_json = wiki_request.json()

wiki_json

{'batchcomplete': '',
 'query': {'pages': {'1095113': {'pageid': 1095113,
    'ns': 0,
    'title': 'Judy Amabile',
    'links': [{'ns': 0, 'title': '2024 ballot measures'},
     {'ns': 0,
      'title': 'Ballot access requirements for political candidates in Colorado'},
     {'ns': 0, 'title': "Ballotpedia's Checks and Balances"},
     {'ns': 0, 'title': "Ballotpedia's Daily Brew"},
     {'ns': 0, 'title': "Ballotpedia's Election Analysis Hub, 2024"},
     {'ns': 0, 'title': "Ballotpedia's Hall Pass"},
     {'ns': 0, 'title': "Ballotpedia's Robe & Gavel"},
     {'ns': 0, 'title': "Ballotpedia's The Ballot Bulletin"},
     {'ns': 0, 'title': "Ballotpedia's Weekly Brew"},
     {'ns': 0, 'title': 'Ballotpedia Email Updates'},
     {'ns': 0, 'title': 'Campaign themes'},
     {'ns': 0, 'title': 'Cities in Colorado'},
     {'ns': 0, 'title': 'Colorado'},
     {'ns': 0, 'title': 'Colorado State Legislature'},
     {'ns': 0, 'title': 'Colorado elections, 2024'},
     {'ns': 0, 'title': 'Color

### Exercise 8: Fix the bug in the `parse_to_links` code above

There is a bug ("TypeError: 'NoneType' object is not subscriptable") in the `parse_to_links` function in the wikifunctions.py code. I have copied the source code for both the `get_page_outlinks` and `parse_to_links` here. Debug what's happening!

In [82]:
from bs4 import BeautifulSoup
from urllib.parse import unquote, quote

def get_page_outlinks(page_title, endpoint='en.wikipedia.org/w/api.php', redirects=1):
    """Takes a page title and returns a list of wiki-links on the page. The 
    list may contain duplicates and the position in the list is approximately 
    where the links occurred.
    
    page_title - a string with the title of the page on Wikipedia
    endpoint - a string that points to the web address of the API.
        This defaults to the English Wikipedia endpoint: 'en.wikipedia.org/w/api.php'
        Changing the two letter language code will return a different language edition
        The Wikia endpoints are slightly different, e.g. 'starwars.wikia.com/api.php'
    redirects - 1 or 0 for whether to follow page redirects, defaults to 1
    
    Returns:
    outlinks_per_lang - a dictionary keyed by language returning a dictionary 
        keyed by page title returning a list of outlinks
    """
    
    # Get the response from the API for a query
    # After passing a page title, the API returns the HTML markup of the current article version within a JSON payload
    #req = requests.get('https://{2}.wikipedia.org/w/api.php?action=parse&format=json&page={0}&redirects={1}&prop=text&disableeditsection=1&disabletoc=1'.format(page_title,redirects,lang))
    query_url = "https://{0}".format(endpoint)
    query_params = {}
    query_params['action'] = 'parse'
    query_params['page'] = page_title
    query_params['redirects'] = redirects
    query_params['prop'] = 'text'
    query_params['disableeditsection'] = 1
    query_params['disabletoc'] = 1
    query_params['format'] = 'json'
    query_params['formatversion'] = 2
    
    json_response = requests.get(url = query_url, params = query_params).json()
    
    if 'parse' in json_response.keys():
        links = parse_to_links(json_response)
        final_title = json_response['parse']['title']
    else:
        links = list()
        final_title = page_title
        
    return links

def parse_to_links(input,is_json=True):
    # Initialize an empty list to store the links
    outlinks_list = []
    
    if is_json:
        page_html = input['parse']['text']#['*']
    else:
        page_html = input

    # Parse the HTML into Beautiful Soup
    soup = BeautifulSoup(page_html,'html.parser')

    # Remove sections at end
    bad_sections = ['See_also','Notes','References','Bibliography','External_links']
    bad_titles = ['Special:','Wikipedia:','Help:','Template:','Category:','International Standard','Portal:','s:','File:','Digital object identifier','(page does not exist)']
    
    sections = soup.find_all('h2')
    for section in sections:
        if section['id'] in bad_sections: ## Error here

            # Clean out the divs
            div_siblings = section.find_next_siblings('div')
            for sibling in div_siblings:
                sibling.clear()

            # Clean out the ULs
            ul_siblings = section.find_next_siblings('ul')
            for sibling in ul_siblings:
                sibling.clear()

    # Delete tags associated with templates
    for tag in soup.find_all('tr'):
        tag.replace_with('')

    # For each paragraph tag, extract the titles within the links
    for para in soup.find_all('p'):
        for link in para.find_all('a'):
            if link.has_attr('title'):
                title = link['title']
                # Ignore links that aren't interesting or are redlinks
                if all(bad not in title for bad in bad_titles) and 'redlink' not in link['href']:
                    outlinks_list.append(title)

    # For each unordered list, extract the titles within the child links
    for unordered_list in soup.find_all('ul'):
        for item in unordered_list.find_all('li'):
            for link in item.find_all('a'):
                if link.has_attr('title'):
                    title = link['title']
                    # Ignore links that aren't interesting or are redlinks
                    if all(bad not in title for bad in bad_titles) and 'redlink' not in link['href']:
                        outlinks_list.append(title)
    
    return outlinks_list

Test it.

In [83]:
get_page_outlinks('University of Colorado Boulder')

['Public university',
 'Research university',
 'Boulder, Colorado',
 'Colorado',
 'Federated state',
 'Flagship university',
 'University of Colorado',
 'Association of American Universities',
 'Carnegie Classification of Institutions of Higher Education',
 'Research I university',
 'National Science Foundation',
 'Colorado Buffaloes',
 'NCAA Division I',
 'Big 12 Conference',
 'NCAA Skiing Championships',
 "NCAA Men's Cross Country Championship",
 "NCAA Women's Cross Country Championship",
 'Colorado Buffaloes football',
 'List of American universities with Olympic medals',
 'Nobel Prize',
 'Pulitzer Prize',
 'MacArthur Fellowship',
 'Turing Award',
 'Astronauts',
 'Associate justice of the United States Supreme Court',
 'Colorado School of Mines',
 'Golden, Colorado',
 'Colorado State University',
 'Fort Collins, Colorado',
 'Cañon City, Colorado',
 'Colorado Department of Corrections',
 'Cañon City, Colorado',
 'High school',
 'World War II',
 'V-12 Navy College Training Program',
 

Debugging.

In [77]:
query_url = "http://en.wikipedia.org/w/api.php"
query_params = {}
query_params['action'] = 'parse'
query_params['page'] = "University of Colorado Boulder"
query_params['redirects'] = 1
query_params['prop'] = 'text'
query_params['disableeditsection'] = 1
query_params['disabletoc'] = 1
query_params['format'] = 'json'
query_params['formatversion'] = 2

json_response = requests.get(url = query_url, params = query_params).json()

In [78]:
soup = BeautifulSoup(json_response['parse']['text'])

sections = soup.find_all('h2')

In [80]:
sections

[<h2 id="History">History</h2>,
 <h2 id="Campus">Campus</h2>,
 <h2 id="Admissions">Admissions</h2>,
 <h2 id="Academics">Academics</h2>,
 <h2 id="Campus_organizations">Campus organizations</h2>,
 <h2 id="Sports,_clubs,_and_traditions"><span id="Sports.2C_clubs.2C_and_traditions"></span>Sports, clubs, and traditions</h2>,
 <h2 id="Demographics">Demographics</h2>,
 <h2 id="Notable_alumni">Notable alumni</h2>,
 <h2 id="Notable_accomplishments_at_CU">Notable accomplishments at CU</h2>,
 <h2 id="Scholarships">Scholarships</h2>,
 <h2 id="See_also">See also</h2>,
 <h2 id="Notes">Notes</h2>,
 <h2 id="References">References</h2>,
 <h2 id="External_links">External links</h2>]

In [79]:
sections[0].span['id']

TypeError: 'NoneType' object is not subscriptable

In [81]:
sections[0]['id']

'History'